# Domain-Adaptive Pretraining (DAPT) 

**Goal**: Pretrain RoBERTa on 62k Brexit/parliamentary text using Masked Language Modeling

**Why Critical**:
- Small labeled dataset (3.3k) → Transfer learning essential
- 62k unlabeled chunks teach Brexit-specific vocabulary & discourse patterns

**Training**: 3-5 epochs, MLM with 15% masking, ~2-3 hours on 6GB GPU

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_NO_TF'] = '1'

# DAPT requires DuckDB to access unlabeled chunks
import duckdb
import pandas as pd
import numpy as np
import torch
from transformers import (
    RobertaTokenizer,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1060
Memory: 6.44 GB


## 1. Load Unlabeled Chunks from Database

**Strategy**: Use all ~62k chunks EXCEPT the ones in training_data_praesentation.csv (labeled train/val/test data)

**Why this approach**:
- DAPT should use UNLABELED data only
- training_data_praesentation.csv contains chunk_ids of all labeled samples (~2,000)
- We exclude these to prevent data leakage
- Remaining ~60-61k chunks are used for unsupervised MLM

In [2]:
# Step 1: Load training_data_praesentation.csv to get chunk_ids of labeled data (to exclude)
print("Loading labeled chunk IDs from training_data_praesentation.csv...")
# Use keep_default_na=False to prevent pandas from converting "None" string to NaN
labeled_chunks = pd.read_csv('training_data_praesentation.csv', keep_default_na=False)

# Check if chunk_id column exists
if 'chunk_id' not in labeled_chunks.columns:
    raise ValueError("❌ 'chunk_id' column not found in training_data_praesentation.csv")
    
labeled_chunk_ids = set(labeled_chunks['chunk_id'].values)
print(f"✅ Found {len(labeled_chunk_ids):,} labeled chunks to EXCLUDE")
print(f"   Columns in training_data_praesentation.csv: {list(labeled_chunks.columns)}")

# Step 2: Connect to DuckDB and load ALL chunks
print("\nConnecting to DuckDB to load all chunks...")
con = duckdb.connect('debates_brexit_chunked.duckdb', read_only=True)

# Query to get total count
total_chunks = con.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"✅ Total chunks in database: {total_chunks:,}")

# Step 3: Load all chunks EXCEPT the labeled ones
# Convert labeled_chunk_ids to a format suitable for SQL IN clause
labeled_ids_str = "','".join(labeled_chunk_ids)
query = f"""
SELECT chunk_id, chunk_text
FROM chunks
WHERE chunk_id NOT IN ('{labeled_ids_str}')
ORDER BY chunk_id
"""

unlabeled_df = con.execute(query).fetchdf()

# Step 4: VERIFY that labeled chunks were actually excluded
print("\n" + "="*80)
print("VERIFYING EXCLUSION OF LABELED CHUNKS")
print("="*80)

# Check if labeled chunk IDs exist in the database
labeled_in_db_query = f"""
SELECT chunk_id
FROM chunks
WHERE chunk_id IN ('{labeled_ids_str}')
"""
labeled_in_db = con.execute(labeled_in_db_query).fetchdf()
labeled_in_db_ids = set(labeled_in_db['chunk_id'].values) if len(labeled_in_db) > 0 else set()

print(f"✅ Verification Results:")
print(f"   Labeled chunks in CSV:           {len(labeled_chunk_ids):,}")
print(f"   Labeled chunks found in DB:      {len(labeled_in_db_ids):,}")
if len(labeled_in_db_ids) < len(labeled_chunk_ids):
    missing = labeled_chunk_ids - labeled_in_db_ids
    print(f"   ⚠️  Warning: {len(missing):,} labeled chunk IDs not found in database")
    if len(missing) <= 10:
        print(f"      Missing IDs: {list(missing)}")
    else:
        print(f"      Sample missing IDs: {list(missing)[:5]}...")

# Verify that NO labeled chunks appear in unlabeled_df
unlabeled_chunk_ids = set(unlabeled_df['chunk_id'].values)
leaked_chunks = labeled_chunk_ids & unlabeled_chunk_ids

if len(leaked_chunks) > 0:
    print(f"\n❌ DATA LEAKAGE DETECTED!")
    print(f"   {len(leaked_chunks):,} labeled chunk IDs found in unlabeled dataset!")
    print(f"   This should NEVER happen - DAPT will be contaminated!")
    if len(leaked_chunks) <= 10:
        print(f"   Leaked chunk IDs: {list(leaked_chunks)}")
    else:
        print(f"   Sample leaked IDs: {list(leaked_chunks)[:5]}...")
    raise ValueError("CRITICAL: Labeled chunks found in unlabeled dataset! DAPT cannot proceed.")
else:
    print(f"   ✅ No labeled chunks in unlabeled dataset (verified)")
    print(f"   ✅ Exclusion successful - {len(labeled_in_db_ids):,} labeled chunks excluded")

con.close()

print("\n" + "="*80)
print("DATA LOADED FOR DOMAIN-ADAPTIVE PRETRAINING (DAPT)")
print("="*80)
print(f"Total chunks in database:     {total_chunks:,}")
print(f"Labeled chunks (excluded):    {len(labeled_in_db_ids):,} (verified in DB)")
print(f"Unlabeled chunks for DAPT:    {len(unlabeled_df):,}")
print(f"                               ({len(unlabeled_df)/total_chunks*100:.1f}% of total)")
print("="*80)
print("\n💡 Correct DAPT approach:")
print("   - Using UNLABELED data only (chunks NOT in training_data_praesentation.csv)")
print("   - Prevents data leakage from labeled train/val/test sets")
print("   - Maximizes unsupervised domain adaptation")
print("   - ✅ VERIFIED: No labeled chunks in unlabeled dataset")
print("="*80)

print(f"\nText length stats:")
print(f"  Mean: {unlabeled_df.chunk_text.str.split().str.len().mean():.0f} words")
print(f"  Median: {unlabeled_df.chunk_text.str.split().str.len().median():.0f} words")

# Sample unlabeled texts
print(f"\nSample unlabeled texts:")
for i in range(min(3, len(unlabeled_df))):
    print(f"  [{i+1}] {unlabeled_df.iloc[i].chunk_text[:100]}...")

Loading labeled chunk IDs from training_data_praesentation.csv...
✅ Found 2,000 labeled chunks to EXCLUDE
   Columns in training_data_praesentation.csv: ['chunk_id', 'chunk_text', 'dataset_split', 'frame_label', 'Annotation']

Connecting to DuckDB to load all chunks...
✅ Total chunks in database: 62,847

VERIFYING EXCLUSION OF LABELED CHUNKS
✅ Verification Results:
   Labeled chunks in CSV:           2,000
   Labeled chunks found in DB:      2,000
   ✅ No labeled chunks in unlabeled dataset (verified)
   ✅ Exclusion successful - 2,000 labeled chunks excluded

DATA LOADED FOR DOMAIN-ADAPTIVE PRETRAINING (DAPT)
Total chunks in database:     62,847
Labeled chunks (excluded):    2,000 (verified in DB)
Unlabeled chunks for DAPT:    60,847
                               (96.8% of total)

💡 Correct DAPT approach:
   - Using UNLABELED data only (chunks NOT in training_data_praesentation.csv)
   - Prevents data leakage from labeled train/val/test sets
   - Maximizes unsupervised domain adaptati

## 2. Initialize RoBERTa Model & Tokenizer

In [3]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 384

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# Load model for Masked Language Modeling
model = RobertaForMaskedLM.from_pretrained(MODEL_NAME)

print(f"✅ Loaded {MODEL_NAME} for MLM")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

✅ Loaded roberta-base for MLM
   Parameters: 124,697,433
   Vocab size: 50,265


## 3. Tokenize Unlabeled Data

In [4]:
# Convert to HuggingFace Dataset
unlabeled_dataset = Dataset.from_pandas(unlabeled_df[['chunk_text']])

def tokenize_function(examples):
    # Tokenize for MLM (no padding here, DataCollator will handle it)
    return tokenizer(
        examples['chunk_text'],
        truncation=True,
        max_length=MAX_LENGTH
    )

print("Tokenizing unlabeled chunks...")
tokenized_dataset = unlabeled_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['chunk_text'],
    desc="Tokenizing"
)

print(f"✅ Tokenized {len(tokenized_dataset):,} chunks")
print(f"   Features: {tokenized_dataset.column_names}")

Tokenizing unlabeled chunks...


Tokenizing:   0%|          | 0/60847 [00:00<?, ? examples/s]

✅ Tokenized 60,847 chunks
   Features: ['input_ids', 'attention_mask']


## 4. Setup Data Collator for MLM

**MLM Strategy**: 15% of tokens are masked for prediction

In [5]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # Standard 15% masking
)

print("✅ Data collator configured:")
print(f"   MLM probability: 15%")
print(f"   Strategy: Random masking of tokens for prediction")

✅ Data collator configured:
   MLM probability: 15%
   Strategy: Random masking of tokens for prediction


## 5. Configure DAPT Training Arguments

In [6]:
training_args = TrainingArguments(
    output_dir="models/roberta-brexit-dapt",
    
    # DAPT-specific settings
    num_train_epochs=3,  # 3-5 epochs typical for DAPT
    
    # Learning rate
    learning_rate=5e-5,  # Slightly higher for pretraining
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    
    # Batch size (REDUCED for 6GB GPU - OOM fix)
    per_device_train_batch_size=4,  # Reduced from 8
    gradient_accumulation_steps=8,  # Increased from 4, maintains effective batch = 32
    
    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,
    
    # Logging
    logging_dir="results/logs_dapt",
    logging_steps=100,
    logging_strategy="steps",
    report_to=["tensorboard"],
    
    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    
    # Reproducibility
    seed=42,
)

print("✅ Training configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   FP16: {training_args.fp16}")

# Estimate training time
steps_per_epoch = len(tokenized_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs
estimated_hours = total_steps * 0.5 / 3600  # ~0.5 sec/step estimate
print(f"\n⏱️  Estimated training time: {estimated_hours:.1f}-{estimated_hours*1.5:.1f} hours")

✅ Training configuration:
   Epochs: 3
   Learning rate: 5e-05
   Batch size: 4
   Gradient accumulation: 8
   Effective batch: 32
   FP16: True

⏱️  Estimated training time: 0.8-1.2 hours


## 6. Initialize Trainer & Start DAPT

In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("\n🚀 Starting Domain-Adaptive Pretraining...")
print("="*80)
print(f"Dataset: {len(tokenized_dataset):,} unlabeled Brexit chunks")
print(f"Objective: Learn Brexit/parliamentary discourse patterns")
print(f"Expected improvement: +5-10% Macro F1 on downstream task")
print("="*80)

# Train
train_result = trainer.train()

print("\n" + "="*80)
print("✅ DAPT COMPLETED!")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds ({train_result.metrics['train_runtime']/3600:.2f} hours)")
print(f"Samples/second: {train_result.metrics['train_samples_per_second']:.2f}")
print("="*80)


🚀 Starting Domain-Adaptive Pretraining...
Dataset: 60,847 unlabeled Brexit chunks
Objective: Learn Brexit/parliamentary discourse patterns
Expected improvement: +5-10% Macro F1 on downstream task


Step,Training Loss
100,1.494500
200,1.346300
300,1.276000
400,1.266100
500,1.265700
600,1.264000
700,1.257700
800,1.246400
900,1.261100
1000,1.234000



✅ DAPT COMPLETED!
Training time: 19818.21 seconds (5.51 hours)
Samples/second: 9.21


## 7. Save Domain-Adapted Model

In [8]:
# Save model and tokenizer
model.save_pretrained("models/roberta-brexit-dapt")
tokenizer.save_pretrained("models/roberta-brexit-dapt")

print("✅ Domain-adapted model saved to models/roberta-brexit-dapt/")
print("\n📂 Saved files:")
print("   - config.json")
print("   - pytorch_model.bin (or model.safetensors)")
print("   - tokenizer files")

# Save training metrics
import pickle
with open("results/dapt_training_results.pkl", "wb") as f:
    pickle.dump(train_result.metrics, f)
print("\n✅ Training metrics saved to results/dapt_training_results.pkl")

✅ Domain-adapted model saved to models/roberta-brexit-dapt/

📂 Saved files:
   - config.json
   - pytorch_model.bin (or model.safetensors)
   - tokenizer files

✅ Training metrics saved to results/dapt_training_results.pkl


## 8. Summary

In [9]:
print("\n" + "="*80)
print("DOMAIN-ADAPTIVE PRETRAINING SUMMARY")
print("="*80)
print(f"\n📊 Training:")
print(f"   - Unlabeled chunks: {len(unlabeled_df):,}")
print(f"   - Epochs: {training_args.num_train_epochs}")
print(f"   - Total time: {train_result.metrics['train_runtime']/3600:.2f} hours")
print(f"\n🎯 Objective: Masked Language Modeling (15% masking)")
print(f"   - Learn Brexit/parliamentary vocabulary")
print(f"   - Adapt to domain-specific discourse patterns")
print(f"\n💾 Saved:")
print(f"   - models/roberta-brexit-dapt/")
print(f"\n📈 Expected Impact:")
print(f"   - Vanilla RoBERTa: ~0.63-0.66 Macro F1")
print(f"   - RoBERTa + DAPT: ~0.70-0.75 Macro F1 (+5-10%)")
print(f"\n✅ Next: 03d_train_model_roberta.ipynb (Classification fine-tuning)")
print("="*80)


DOMAIN-ADAPTIVE PRETRAINING SUMMARY

📊 Training:
   - Unlabeled chunks: 60,847
   - Epochs: 3
   - Total time: 5.51 hours

🎯 Objective: Masked Language Modeling (15% masking)
   - Learn Brexit/parliamentary vocabulary
   - Adapt to domain-specific discourse patterns

💾 Saved:
   - models/roberta-brexit-dapt/

📈 Expected Impact:
   - Vanilla RoBERTa: ~0.63-0.66 Macro F1
   - RoBERTa + DAPT: ~0.70-0.75 Macro F1 (+5-10%)

✅ Next: 03d_train_model_roberta.ipynb (Classification fine-tuning)
